# Superposition: more features than dimensions
1. Random directions crowd near 90 degrees, more tightly as the dimension grows (spread about 57.3 / sqrt(d) degrees).
2. A limit: N unit vectors in d dimensions have mean squared cosine at least (N/d - 1)/(N - 1).
3. Optimisation: 10,000 vectors in 100 dimensions nudged to be more perpendicular; histogram at every step.
4. Johnson-Lindenstrauss (Dasgupta & Gupta 2003): dimension needed for n nearly perpendicular directions.
5. Real angles between rows of GPT-2 small's token-embedding table.
6. The toy model of superposition (Elhage et al. 2022): 5 features, 2 hidden numbers, ReLU output, several sparsities.
Heavy part (3) takes a few minutes; run on a machine with spare cores.

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT.parents[2] / "tools"))
import gpt2_small as g
np.set_printoptions(precision=4, suppress=True)
deg = lambda c: np.degrees(np.arccos(np.clip(c, -1, 1)))


def unit_rows(n, d, rng):
    X = rng.normal(size=(n, d)).astype(np.float32)
    return X / np.linalg.norm(X, axis=1, keepdims=True)

## 1. Angles between random directions, by dimension
20,000 random pairs per dimension. Prediction from the derivation in the Note: the cosine has mean 0 and variance 1/d, so the angle's spread is about (180/pi)/sqrt(d) degrees.

In [ ]:
rng = np.random.default_rng(0)
DIMS = [2, 3, 10, 30, 100, 300, 1000, 12288]
rows, hist = [], []
bins = np.arange(0, 181, 1.0)
for d in DIMS:
    a, b = unit_rows(20_000, d, rng), unit_rows(20_000, d, rng)
    c = (a * b).sum(1)
    ang = deg(c)
    rows.append((d, c.mean(), (c ** 2).mean(), 1 / d, ang.std(), 180 / np.pi / np.sqrt(d),
                 (abs(ang - 90) <= 1).mean(), (abs(ang - 90) <= 5).mean()))
    hist.append(np.histogram(ang, bins)[0] / len(ang))
dims = pd.DataFrame(rows, columns=["d", "mean_cos", "mean_cos2", "predicted_mean_cos2", "angle_std_deg",
                                   "predicted_std_deg", "share_89_91", "share_85_95"])
print(dims.round(4).to_string())
dims.round(5).to_csv(DATA / "random_angles_summary.csv", index=False)
pd.DataFrame(np.array(hist).T, columns=[str(d) for d in DIMS]).assign(angle=bins[:-1] + 0.5).round(6) \
  .to_csv(DATA / "random_angles_hist.csv", index=False)

## 2. How perpendicular can N directions be? (trace bound, derived in the Note)

In [ ]:
def mean_cos2_bound(N, d):
    return (N / d - 1) / (N - 1)

for N, d in [(10_000, 100), (200, 100), (101, 100)]:
    b = mean_cos2_bound(N, d)
    print(f"N={N}, d={d}: mean cos^2 >= {b:.5f}, RMS cos >= {np.sqrt(b):.4f}, "
          f"i.e. some pair at least {90 - deg(np.sqrt(b)):.2f} degrees away from 90")
# Check: random vectors sit just above the bound
Ur = unit_rows(2000, 50, rng)
G = Ur @ Ur.T
print("random 2000 x 50: mean off-diagonal cos^2 = %.5f, bound %.5f"
      % (((G ** 2).sum() - 2000) / (2000 * 1999), mean_cos2_bound(2000, 50)))

## 3. Optimisation: 10,000 vectors in 100 dimensions
Loss = sum over pairs of cos^8 (a high power pushes hardest on the pairs furthest from 90 degrees). Each step moves every vector against its gradient along the sphere, then rescales it to length 1.

In [ ]:
N, d, STEPS = 10_000, 100, 60
U = unit_rows(N, d, np.random.default_rng(1))
abins = np.arange(50, 130.01, 0.5)
iu = np.triu_indices(N, 1)
opt_hist, opt_stats = [], []
t0 = time.time()
for s in range(STEPS + 1):
    G = U @ U.T
    np.fill_diagonal(G, 0)
    c = G[iu]
    opt_hist.append(np.histogram(deg(c), abins)[0] / len(c))
    opt_stats.append((s, 90 - deg(np.abs(c).max()), np.sqrt((c ** 2).mean()), (abs(c) <= np.sin(np.radians(1))).mean()))
    if s == STEPS:
        break
    grad = (G ** 7) @ U
    grad -= (grad * U).sum(1, keepdims=True) * U                 # keep only the part along the sphere
    U -= 0.01 * grad / np.abs(grad).max()
    U /= np.linalg.norm(U, axis=1, keepdims=True)
del G, c
stats = pd.DataFrame(opt_stats, columns=["step", "worst_deg_from_90", "rms_cos", "share_89_91"])
print(stats.iloc[[0, 1, 2, 5, 10, 20, 40, 60]].round(4).to_string(), f"\n{time.time() - t0:.0f} s")
print("bound for RMS cos: %.4f" % np.sqrt(mean_cos2_bound(N, d)))
stats.round(5).to_csv(DATA / "optimise_stats.csv", index=False)
pd.DataFrame(np.array(opt_hist).T, columns=[str(s) for s in range(STEPS + 1)]) \
  .assign(angle=abins[:-1] + 0.25).round(7).to_csv(DATA / "optimise_hist.csv", index=False)

## 4. Johnson-Lindenstrauss: dimension k needed for n points (Dasgupta & Gupta 2003, Theorem 2.1)
k >= 4 ln(n) / (eps^2/2 - eps^3/3). Applied to the n standard basis vectors plus the origin (derivation in the Note), every pair of images has |cos| <= 2 eps / (1 - eps).

In [ ]:
def jl_k(n, eps):
    return int(np.ceil(4 * np.log(n) / (eps ** 2 / 2 - eps ** 3 / 3)))

eps = 0.1
print("eps = 0.1 -> |cos| <= %.3f, angles within %.1f degrees of 90" % (2 * eps / (1 - eps), 90 - deg(2 * eps / (1 - eps))))
jl = pd.DataFrame([(n, jl_k(n + 1, eps)) for n in (10 ** 3, 10 ** 6, 10 ** 9, 10 ** 12)], columns=["directions", "k_needed"])
print(jl.to_string())
jl.to_csv(DATA / "jl_table.csv", index=False)
# What random directions do in practice: worst angle among 10,000 random directions in 768 and 12,288 dimensions
for dd in (768, 12288):
    R = unit_rows(10_000, dd, np.random.default_rng(2))
    Gr = R @ R.T
    np.fill_diagonal(Gr, 0)
    print(f"10,000 random directions in {dd} dims: worst pair {90 - deg(np.abs(Gr).max()):.1f} degrees from 90")
del R, Gr

## 5. Real angles: rows of GPT-2 small's token-embedding table (768 numbers per token)

In [ ]:
W = g.load_wte()
ids = np.random.default_rng(3).choice(len(W), 3000, replace=False)
X = W[ids]
out = []
for name, M in (("raw", X), ("centred", X - W.mean(0))):
    Mu = M / np.linalg.norm(M, axis=1, keepdims=True)
    c = (Mu @ Mu.T)[np.triu_indices(3000, 1)]
    a = deg(c)
    print(f"GPT-2 {name}: mean angle {a.mean():.1f}, std {a.std():.1f}, share within 85-95: {(abs(a - 90) <= 5).mean():.3f}")
    out.append(pd.Series(np.histogram(a, bins)[0] / len(a), name=name))
R = unit_rows(3000, 768, np.random.default_rng(4))
c = (R @ R.T)[np.triu_indices(3000, 1)]
print(f"random 768-d: mean angle {deg(c).mean():.1f}, std {deg(c).std():.1f}")
out.append(pd.Series(np.histogram(deg(c), bins)[0] / len(c), name="random"))
pd.concat(out, axis=1).assign(angle=bins[:-1] + 0.5).round(6).to_csv(DATA / "gpt2_angles_hist.csv", index=False)

## 6. The toy model of superposition (Elhage et al. 2022)
n = 5 features, m = 2 hidden numbers. Each feature is 0 with probability S, else uniform on [0, 1]. Model: x' = ReLU(W^T W x + b); loss = sum_i I_i (x_i - x'_i)^2. Adam, 6,000 steps of 1,024 samples; 5 seeds per setting, keep the lowest final loss (the paper keeps the best of 1,000).

In [ ]:
def train(S, I, relu=True, n=5, m=2, steps=6000, batch=1024, lr=1e-2, seed=0, snap_at=()):
    r = np.random.default_rng(seed)
    W = r.normal(0, np.sqrt(2 / (n + m)), (m, n))
    b = np.zeros(n)
    mom = [np.zeros_like(W), np.zeros_like(b)]
    vel = [np.zeros_like(W), np.zeros_like(b)]
    snaps = []
    for t in range(1, steps + 1):
        x = r.uniform(0, 1, (batch, n)) * (r.uniform(0, 1, (batch, n)) > S)
        z = x @ W.T @ W + b
        y = np.maximum(z, 0) if relu else z
        gz = 2 * I * (y - x) * ((z > 0) if relu else 1) / batch           # dLoss/dz
        grads = [W @ (x.T @ gz + gz.T @ x), gz.sum(0)]
        for p, gp, mo, ve in zip((W, b), grads, mom, vel):              # Adam
            mo *= 0.9; mo += 0.1 * gp
            ve *= 0.999; ve += 0.001 * gp ** 2
            p -= lr * (mo / (1 - 0.9 ** t)) / (np.sqrt(ve / (1 - 0.999 ** t)) + 1e-8)
        if t in snap_at:
            snaps.append((t, W.copy()))
    x = r.uniform(0, 1, (100_000, n)) * (r.uniform(0, 1, (100_000, n)) > S)
    z = x @ W.T @ W + b
    loss = (I * ((np.maximum(z, 0) if relu else z) - x) ** 2).sum(1).mean()
    return W, b, loss, snaps


def best(S, I, relu=True):
    return min((train(S, I, relu, seed=s) for s in range(5)), key=lambda r: r[2])

I_decay = 0.7 ** np.arange(5)
cases = [("ReLU", "0.7^i", S, *best(S, I_decay)[:3]) for S in (0.0, 0.7, 0.9, 0.95)]
cases.append(("linear", "0.7^i", 0.95, *best(0.95, I_decay, relu=False)[:3]))
cases.append(("ReLU", "1", 0.95, *best(0.95, np.ones(5))[:3]))
rows = []
for model, imp, S, W, b, loss in cases:
    norms = np.linalg.norm(W, axis=0)
    ang = np.degrees(np.arctan2(W[1], W[0]))
    print(f"{model:6s} I={imp:5s} S={S:.2f}: loss {loss:.4f}, norms {norms.round(2)}, "
          f"represented (norm > 0.5): {(norms > 0.5).sum()}, angles {ang.round(0)}")
    for i in range(5):
        rows.append((model, imp, S, i + 1, W[0, i], W[1, i], norms[i], b[i], loss))
toy = pd.DataFrame(rows, columns=["model", "importance", "S", "feature", "w1", "w2", "norm", "bias", "loss"])
toy.round(4).to_csv(DATA / "toy_models.csv", index=False)
u = toy[(toy.importance == "1")]
a = np.sort(np.degrees(np.arctan2(u.w2, u.w1)) % 360)
print("uniform case, gaps between neighbouring feature directions:", np.diff(np.r_[a, a[0] + 360]).round(1))

Training run of the uniform case (I = 1, S = 0.95), for the animation: W every 10 steps up to 300, then every 200; seed of the best run.

In [ ]:
seed = min(range(5), key=lambda s: train(0.95, np.ones(5), seed=s)[2])
_, _, _, snaps = train(0.95, np.ones(5), seed=seed, snap_at=set([1] + list(range(10, 301, 10)) + list(range(400, 6001, 200))))
pd.DataFrame([(t, i + 1, Wt[0, i], Wt[1, i]) for t, Wt in snaps for i in range(5)],
             columns=["step", "feature", "w1", "w2"]).round(4).to_csv(DATA / "toy_training.csv", index=False)
print("seed", seed, "snapshots", len(snaps))